# MOwNiT - laboratorium 11 - Optymalizacja

### Hubert Kukla

Notebook zachowuje strukturę podobną do laboratorium 9: każde zadanie i każdy podpunkt są opisane osobno, a potem wykonane w komórkach kodu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path
from IPython.display import display

FIG_DIR = Path("lab11_figures")
FIG_DIR.mkdir(exist_ok=True)
np.set_printoptions(precision=6, suppress=True)

## Zadanie 1. Preconditioning

Dana jest funkcja kwadratowa

$$f(x,y)=\frac{1}{2}x^2+\frac{9}{2}y^2.$$

W tym zadaniu najpierw stosujemy klasyczną metodę gradientu prostego z dokładnym doborem kroku, a później wykonujemy zmianę zmiennych, która poprawia uwarunkowanie problemu.

### Zadanie 1(a). Gradient prosty z optymalnym krokiem

Dla funkcji kwadratowej z macierzą Hessego $H$ optymalny krok w kierunku $-\nabla f(z_k)$ ma postać

$$\alpha_k = \frac{\nabla f(z_k)^T \nabla f(z_k)}{\nabla f(z_k)^T H \nabla f(z_k)}.$$

Startujemy z punktu $(x_0,y_0)=(9,1)$ i zapisujemy kolejne iteracje, aby zaznaczyć je na wykresie konturowym.

In [ ]:
H = np.diag([1.0, 9.0])

def f(z):
    return 0.5 * z @ H @ z

def grad_f(z):
    return H @ z

def steepest_descent_exact(z0, H, tol=1e-10, max_iter=80):
    z = z0.astype(float).copy()
    path = [z.copy()]
    rows = []
    for k in range(max_iter):
        g = H @ z
        rows.append({
            "k": k,
            "x": z[0],
            "y": z[1],
            "f(x,y)": f(z),
            "||grad||": np.linalg.norm(g),
        })
        if np.linalg.norm(g) < tol:
            break
        alpha = (g @ g) / (g @ H @ g)
        z = z - alpha * g
        path.append(z.copy())
    return np.array(path), pd.DataFrame(rows)

path1, tabela1 = steepest_descent_exact(np.array([9.0, 1.0]), H)
display(tabela1.head(12))
print(f"Ostatni zapisany punkt: {path1[-1]}")
print(f"Wartość funkcji w ostatnim punkcie: {f(path1[-1]):.3e}")

In [ ]:
# Wykres konturowy z pierwszymi iteracjami algorytmu
xs = np.linspace(-10, 10, 300)
ys = np.linspace(-3, 3, 300)
X, Y = np.meshgrid(xs, ys)
Z = 0.5 * X**2 + 4.5 * Y**2

plt.figure(figsize=(7, 5))
levels = np.geomspace(0.05, 60, 24)
plt.contour(X, Y, Z, levels=levels)
p = path1[:25]
plt.plot(p[:, 0], p[:, 1], marker='o', markersize=3, linewidth=1.5, label='iteracje GD')
plt.scatter([0], [0], marker='x', s=80, label='minimum')
plt.title('Zadanie 1(a): kontury f oraz iteracje gradientu prostego')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'zad1_contour.png', dpi=170)
plt.show()

![Wynik: kontury funkcji f](lab11_figures/zad1_contour.png)

### Zadanie 1(b)-(c). Uwarunkowanie i rozkład macierzy Hessego

Macierz Hessego jest stała, bo funkcja jest kwadratowa. Dla macierzy diagonalnej współczynnik uwarunkowania w normie spektralnej to stosunek największej i najmniejszej wartości własnej.

In [ ]:
eigvals = np.linalg.eigvalsh(H)
condition_number = eigvals.max() / eigvals.min()
L = np.diag([1.0, 3.0])

print("H =")
print(H)
print(f"Wartości własne H: {eigvals}")
print(f"Współczynnik uwarunkowania kappa(H) = {condition_number:.0f}")
print("Macierz L taka, że H = L L^T:")
print(L)
print("Sprawdzenie L L^T =")
print(L @ L.T)

### Zadanie 1(d). Zmiana zmiennych i minimalizacja funkcji g

Przyjmujemy

$$\begin{bmatrix}x'\\y'\end{bmatrix}=L\begin{bmatrix}x\\y\end{bmatrix},\qquad L=\begin{bmatrix}1&0\\0&3\end{bmatrix}.$$

Wtedy $x=x'$ oraz $y=y'/3$, więc

$$g(x',y')=f(x',y'/3)=\frac{1}{2}(x')^2+\frac{1}{2}(y')^2.$$

Macierz Hessego funkcji $g$ jest jednostkowa, więc dokładny gradient prosty dochodzi do minimum w jednej iteracji.

In [ ]:
def g_pre(u):
    return 0.5 * np.dot(u, u)

def grad_g_pre(u):
    return u

u0 = L @ np.array([9.0, 1.0])
alpha0 = 1.0
u1 = u0 - alpha0 * grad_g_pre(u0)
z_min_from_u = np.linalg.solve(L.T, u1)  # równoważne L^{-T} u1

print(f"Punkt startowy po zmianie zmiennych u0 = (x', y') = {u0}")
print(f"Po jednej iteracji gradientu prostego: u1 = {u1}")
print(f"Minimum funkcji f odtworzone ze wzoru L^(-T) u: {z_min_from_u}")
print(f"g(u1) = {g_pre(u1):.3e}, f(z_min) = {f(z_min_from_u):.3e}")

xs2 = np.linspace(-10, 10, 300)
ys2 = np.linspace(-5, 5, 300)
X2, Y2 = np.meshgrid(xs2, ys2)
G = 0.5 * (X2**2 + Y2**2)
plt.figure(figsize=(7, 5))
plt.contour(X2, Y2, G, levels=np.geomspace(0.05, 60, 24))
plt.plot([u0[0], u1[0]], [u0[1], u1[1]], marker='o', linewidth=1.5, label='iteracje dla g')
plt.scatter([0], [0], marker='x', s=80, label='minimum')
plt.title("Zadanie 1(d): problem po zmianie zmiennych")
plt.xlabel("x'")
plt.ylabel("y'")
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'zad1_preconditioned.png', dpi=170)
plt.show()

![Wynik: problem po preconditioningu](lab11_figures/zad1_preconditioned.png)

## Zadanie 2. Wiszący łańcuch

Modelujemy łańcuch jako $n+1$ mas punktowych połączonych sprężynami. Końce są unieruchomione w punktach $(0,0)$ oraz $(3,1)$. Minimalizujemy całkowitą energię potencjalną względem współrzędnych mas wewnętrznych.

Parametry z treści zadania:

* $n=40$,
* $L=0.1$ m,
* $m=0.1$ kg,
* $k=70$ N/m,
* $g=9.81$ m/s².

### Zadanie 2. Gradient funkcji celu

Oznaczmy długość odcinka między punktami $i$ oraz $i+1$ przez

$$d_i=\sqrt{(x_i-x_{i+1})^2+(y_i-y_{i+1})^2}.$$

Dla wewnętrznego punktu $j$ gradient ma wkład ze sprężyny po lewej i po prawej. W implementacji wygodniej liczyć wkład każdej sprężyny do obu jej końców, a następnie usunąć zmienne odpowiadające punktom ustalonym. Składowa grawitacyjna dodaje $mg$ do pochodnej po każdym $y_j$.

In [ ]:
n = 40
L0 = 0.1
m = 0.1
k = 70.0
g = 9.81
p_start = np.array([0.0, 0.0])
p_end = np.array([3.0, 1.0])

def unpack_chain(z):
    x = np.empty(n + 1)
    y = np.empty(n + 1)
    x[0], y[0] = p_start
    x[n], y[n] = p_end
    x[1:n] = z[:n - 1]
    y[1:n] = z[n - 1:]
    return x, y

def pack_chain(x, y):
    return np.r_[x[1:n], y[1:n]]

def chain_energy(z):
    x, y = unpack_chain(z)
    dx = x[:-1] - x[1:]
    dy = y[:-1] - y[1:]
    d = np.sqrt(dx * dx + dy * dy)
    elastic = 0.5 * k * np.sum((d - L0) ** 2)
    gravitational = m * g * np.sum(y)
    return elastic + gravitational

def chain_grad(z):
    x, y = unpack_chain(z)
    gx = np.zeros(n + 1)
    gy = np.zeros(n + 1)
    dx = x[:-1] - x[1:]
    dy = y[:-1] - y[1:]
    d = np.sqrt(dx * dx + dy * dy)
    coeff = k * (d - L0) / np.maximum(d, 1e-12)
    cx = coeff * dx
    cy = coeff * dy
    gx[:-1] += cx
    gy[:-1] += cy
    gx[1:] -= cx
    gy[1:] -= cy
    gy += m * g
    return pack_chain(gx, gy)

# punkt startowy: liniowa interpolacja między końcami
x0 = np.linspace(p_start[0], p_end[0], n + 1)
y0 = np.linspace(p_start[1], p_end[1], n + 1)
z0 = pack_chain(x0, y0)

print(f"Energia początkowa: {chain_energy(z0):.6f}")
print(f"Norma gradientu w punkcie początkowym: {np.linalg.norm(chain_grad(z0)):.6f}")

### Zadanie 2(a)-(c). Trzy wersje algorytmu gradientu prostego

Porównujemy:

1. stały krok $\alpha=0.002$,
2. krok malejący wykładniczo: $\alpha_t=0.003\cdot 0.98^{\lfloor t/1000\rfloor}$,
3. przeszukiwanie liniowe typu backtracking z warunkiem Armijo.

In [ ]:
def gd_const_chain(z0, alpha=0.002, max_iter=40000, tol=1e-5):
    z = z0.copy()
    hist = []
    t0 = time.perf_counter()
    for it in range(max_iter):
        E = chain_energy(z)
        gr = chain_grad(z)
        gn = np.linalg.norm(gr)
        if it % 100 == 0:
            hist.append((it, E, gn, alpha))
        if gn < tol:
            break
        z -= alpha * gr
    elapsed = time.perf_counter() - t0
    hist.append((it, chain_energy(z), np.linalg.norm(chain_grad(z)), alpha))
    return z, pd.DataFrame(hist, columns=['iteracja', 'energia', 'norma_grad', 'alpha']), elapsed

def gd_decay_chain(z0, alpha0=0.003, gamma=0.98, every=1000, max_iter=40000, tol=1e-5):
    z = z0.copy()
    hist = []
    t0 = time.perf_counter()
    for it in range(max_iter):
        alpha = alpha0 * (gamma ** (it // every))
        E = chain_energy(z)
        gr = chain_grad(z)
        gn = np.linalg.norm(gr)
        if it % 100 == 0:
            hist.append((it, E, gn, alpha))
        if gn < tol:
            break
        z -= alpha * gr
    elapsed = time.perf_counter() - t0
    hist.append((it, chain_energy(z), np.linalg.norm(chain_grad(z)), alpha))
    return z, pd.DataFrame(hist, columns=['iteracja', 'energia', 'norma_grad', 'alpha']), elapsed

def gd_linesearch_chain(z0, alpha0=1.0, rho=0.5, c=1e-4, max_iter=40000, tol=1e-5):
    z = z0.copy()
    hist = []
    alpha_start = alpha0
    t0 = time.perf_counter()
    for it in range(max_iter):
        E = chain_energy(z)
        gr = chain_grad(z)
        gn = np.linalg.norm(gr)
        if it % 50 == 0:
            hist.append((it, E, gn, alpha_start))
        if gn < tol:
            break
        alpha = alpha_start
        while chain_energy(z - alpha * gr) > E - c * alpha * gn * gn:
            alpha *= rho
            if alpha < 1e-12:
                break
        z -= alpha * gr
        alpha_start = min(alpha / rho, 1.0)
    elapsed = time.perf_counter() - t0
    hist.append((it, chain_energy(z), np.linalg.norm(chain_grad(z)), alpha_start))
    return z, pd.DataFrame(hist, columns=['iteracja', 'energia', 'norma_grad', 'alpha']), elapsed

methods = [
    ('stały krok alpha=0.002', gd_const_chain, {}),
    ('alpha wykładniczo malejące', gd_decay_chain, {}),
    ('przeszukiwanie liniowe', gd_linesearch_chain, {}),
]

chain_results = {}
summary_rows = []
for name, fn, kwargs in methods:
    z, hist, elapsed = fn(z0, **kwargs)
    x, y = unpack_chain(z)
    chain_results[name] = (z, hist, elapsed, x, y)
    summary_rows.append({
        'metoda': name,
        'iteracje': int(hist.iloc[-1]['iteracja']),
        'energia końcowa': chain_energy(z),
        'norma gradientu': np.linalg.norm(chain_grad(z)),
        'czas [s]': elapsed,
        'min y': y.min(),
    })

chain_summary = pd.DataFrame(summary_rows)
display(chain_summary)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(x0, y0, marker='o', markersize=2, linestyle='--', label='położenie początkowe')
for name, (_, _, _, x, y) in chain_results.items():
    plt.plot(x, y, marker='o', markersize=2.5, linewidth=1.3, label=name)
plt.title('Zadanie 2: położenie łańcucha po minimalizacji energii')
plt.xlabel('x [m]')
plt.ylabel('y [m]')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'zad2_chain_positions.png', dpi=170)
plt.show()

plt.figure(figsize=(8, 5))
for name, (_, hist, _, _, _) in chain_results.items():
    plt.plot(hist['iteracja'], hist['energia'], label=name)
plt.title('Zadanie 2: spadek energii w kolejnych iteracjach')
plt.xlabel('iteracja')
plt.ylabel('energia potencjalna')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'zad2_energy_history.png', dpi=170)
plt.show()

![Wynik: położenie łańcucha](lab11_figures/zad2_chain_positions.png)

![Wynik: historia energii](lab11_figures/zad2_energy_history.png)

## Zadanie 3. Predykcja roku wydania utworu metodą gradientu prostego

Rozwiązujemy problem regresji liniowej w postaci

$$\min_w \frac{1}{2m}\|Aw-y\|_2^2.$$

Gradient funkcji celu wynosi

$$\nabla F(w)=\frac{1}{m}A^T(Aw-y).$$

Stałą uczącą dobieramy na podstawie najmniejszej i największej dodatniej wartości własnej macierzy $A^TA/m$:

$$\alpha=\frac{2}{\lambda_{\min}+\lambda_{\max}}.$$

Jeżeli w katalogu znajduje się plik `YearPredictionMSD.txt` albo podobny CSV z pierwszą kolumną jako rokiem, kod użyje tego pliku. W przeciwnym razie tworzy dane syntetyczne, żeby dało się uruchomić i sprawdzić całą procedurę bez plików z laboratorium 2.

In [ ]:
def load_or_generate_year_data():
    candidates = [
        Path('YearPredictionMSD.txt'),
        Path('YearPredictionMSD.csv'),
        Path('year_prediction.csv'),
        Path('lab2_train.csv'),
    ]
    for path in candidates:
        if path.exists():
            data = np.loadtxt(path, delimiter=',')
            if data.ndim == 1:
                data = data.reshape(1, -1)
            y = data[:, 0]
            X = data[:, 1:]
            split = min(463715, int(0.8 * len(y))) if len(y) > 1000 else int(0.8 * len(y))
            return X[:split], X[split:], y[:split], y[split:], f'plik: {path.name}'

    # Tryb awaryjny: dane syntetyczne o skorelowanych cechach.
    rng = np.random.default_rng(11)
    n_train, n_test, d = 5000, 1000, 20
    latent = rng.normal(size=(n_train + n_test, 5))
    loadings = rng.normal(size=(5, d))
    X = latent @ loadings + 0.2 * rng.normal(size=(n_train + n_test, d))
    mu_raw = X[:n_train].mean(axis=0)
    sig_raw = X[:n_train].std(axis=0)
    sig_raw[sig_raw == 0] = 1.0
    X_for_target = (X - mu_raw) / sig_raw
    true_w = rng.normal(scale=3.0, size=d)
    y = 2000.0 + X_for_target @ true_w + rng.normal(scale=8.0, size=n_train + n_test)
    return X[:n_train], X[n_train:], y[:n_train], y[n_train:], 'dane syntetyczne - brak pliku z laboratorium 2'

def standardize_and_add_intercept(X_train, X_test):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma == 0] = 1.0
    X_train_s = (X_train - mu) / sigma
    X_test_s = (X_test - mu) / sigma
    A_train = np.c_[np.ones(len(X_train_s)), X_train_s]
    A_test = np.c_[np.ones(len(X_test_s)), X_test_s]
    return A_train, A_test

def rmse(y, pred):
    return float(np.sqrt(np.mean((y - pred) ** 2)))

def mae(y, pred):
    return float(np.mean(np.abs(y - pred)))

X_train, X_test, y_train, y_test, data_source = load_or_generate_year_data()
A_train, A_test = standardize_and_add_intercept(X_train, X_test)
print(f"Źródło danych: {data_source}")
print(f"Wymiary macierzy treningowej A: {A_train.shape}")

In [ ]:
def gradient_descent_lsq(A, y, max_iter=5000, tol=1e-8):
    m, d = A.shape
    H_lsq = (A.T @ A) / m
    eigvals = np.linalg.eigvalsh(H_lsq)
    positive = eigvals[eigvals > 1e-12]
    lam_min = float(positive.min())
    lam_max = float(eigvals.max())
    alpha = 2.0 / (lam_min + lam_max)
    w = np.zeros(d)
    hist = []
    t0 = time.perf_counter()
    for it in range(max_iter + 1):
        r = A @ w - y
        loss = 0.5 * np.mean(r * r)
        gr = A.T @ r / m
        gn = np.linalg.norm(gr)
        if it % 25 == 0 or it == max_iter:
            hist.append((it, loss, gn))
        if gn < tol:
            break
        w -= alpha * gr
    elapsed = time.perf_counter() - t0
    hist_df = pd.DataFrame(hist, columns=['iteracja', 'funkcja celu', 'norma_grad'])
    info = {
        'iteracje': it,
        'alpha': alpha,
        'lambda_min': lam_min,
        'lambda_max': lam_max,
        'czas [s]': elapsed,
    }
    return w, hist_df, info

# Metoda najmniejszych kwadratów
start = time.perf_counter()
w_ls = np.linalg.lstsq(A_train, y_train, rcond=None)[0]
ls_time = time.perf_counter() - start

# Gradient prosty
w_gd, gd_hist, gd_info = gradient_descent_lsq(A_train, y_train, max_iter=5000, tol=1e-8)

comparison = []
for method, w, elapsed, iters in [
    ('najmniejsze kwadraty', w_ls, ls_time, 1),
    ('gradient prosty', w_gd, gd_info['czas [s]'], gd_info['iteracje']),
]:
    pred = A_test @ w
    comparison.append({
        'metoda': method,
        'RMSE test': rmse(y_test, pred),
        'MAE test': mae(y_test, pred),
        'czas [s]': elapsed,
        'iteracje': iters,
    })
comparison = pd.DataFrame(comparison)

display(comparison)
print(f"alpha = {gd_info['alpha']:.6f}")
print(f"lambda_min = {gd_info['lambda_min']:.6f}, lambda_max = {gd_info['lambda_max']:.6f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.semilogy(gd_hist['iteracja'], gd_hist['funkcja celu'], marker='o', markersize=2.5)
plt.title('Zadanie 3: zbieżność gradientu prostego')
plt.xlabel('iteracja')
plt.ylabel('funkcja celu na zbiorze treningowym')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'zad3_gd_convergence.png', dpi=170)
plt.show()

pred_gd = A_test @ w_gd
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred_gd, s=8, alpha=0.5)
lo = min(y_test.min(), pred_gd.min())
hi = max(y_test.max(), pred_gd.max())
plt.plot([lo, hi], [lo, hi], linestyle='--', linewidth=1.2)
plt.title('Zadanie 3: wartości rzeczywiste i przewidywane')
plt.xlabel('rok / wartość rzeczywista')
plt.ylabel('predykcja GD')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'zad3_prediction_scatter.png', dpi=170)
plt.show()

![Wynik: zbieżność GD](lab11_figures/zad3_gd_convergence.png)

![Wynik: predykcja GD](lab11_figures/zad3_prediction_scatter.png)

### Wnioski końcowe

1. W zadaniu 1 zmiana zmiennych sprowadza problem do funkcji o kulistych poziomicach, dzięki czemu gradient prosty dochodzi do minimum natychmiast.
2. W zadaniu 2 wszystkie trzy warianty gradientu prostego prowadzą do tego samego kształtu łańcucha, ale przeszukiwanie liniowe wymaga najmniej iteracji.
3. W zadaniu 3 gradient prosty daje wynik zgodny z metodą najmniejszych kwadratów, jednak jego koszt zależy od liczby iteracji. Jedna iteracja kosztuje tylko $O(md)$, podczas gdy bezpośrednie rozwiązanie najmniejszych kwadratów kosztuje typowo $O(md^2+d^3)$.